In [ ]:
import os
import subprocess

# 1. Clone the repository
!git clone --depth 1 https://github.com/debpalash/VoiceStudio.git /kaggle/working/VoiceStudio
%cd /kaggle/working/VoiceStudio

# 2. Install system audio libraries
!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

# 3. Install Bun (JavaScript runtime for building the Web UI)
!curl -fsSL https://bun.sh/install | bash
os.environ["PATH"] = f"/root/.bun/bin:{os.environ['PATH']}"
os.environ["UV_HTTP_TIMEOUT"] = "300"

# 4. Install uv (fast Python package manager)
!pip install -q uv

# 5. Install Cloudflared for exposing the Web UI port
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("Cell 1 Complete: System tools and repository are ready.")

In [ ]:
import os
import sys

%cd /kaggle/working/VoiceStudio
os.environ["PATH"] = f"/root/.bun/bin:{os.environ['PATH']}"

# 1. Build the Electron Web Renderer (Frontend UI)
!bun install --frozen-lockfile
!bun run --cwd electron build:web

# 2. Install Python backend dependencies with PyTorch version constraints
!uv pip install --system --no-cache --constraint deploy/torch-constraints.txt .

# 3. Setup cuDNN 8 compatibility for GPU speech-to-text (WhisperX / CTranslate2)
os.makedirs("/kaggle/working/VoiceStudio/.venv", exist_ok=True)
!python3 scripts/setup.py

print("Cell 2 Complete: Frontend built and Python backend installed.")

In [ ]:
# 1. Patch backend/api/dependencies.py to allow non-loopback requests
dep_file = "/kaggle/working/VoiceStudio/backend/api/dependencies.py"
with open(dep_file, "r") as f:
    dep_code = f.read()

dep_code = dep_code.replace(
    'raise HTTPException(status_code=403, detail="loopback origin required")',
    "return None",
)
dep_code = dep_code.replace(
    'raise HTTPException(status_code=403, detail="desktop origin required")',
    "return None",
)
dep_code = dep_code.replace(
    'def _admin_gate_403() -> None:\n    """',
    'def _admin_gate_403() -> None:\n    return None\n    """',
)

with open(dep_file, "w") as f:
    f.write(dep_code)

# 2. Patch backend/core/browser_guard.py to allow external tunnel hostnames
guard_file = "/kaggle/working/VoiceStudio/backend/core/browser_guard.py"
with open(guard_file, "r") as f:
    guard_code = f.read()

guard_code = guard_code.replace(
    "detail = self._refusal(scope, connection, method)", "detail = None"
)
guard_code = guard_code.replace(
    "raise HTTPException(status_code=403, detail=CROSS_SITE_DETAIL)",
    "return None",
)

with open(guard_file, "w") as f:
    f.write(guard_code)

print("Cell 3 Complete: Cloud tunnel access unlocked.")

In [ ]:
from huggingface_hub import snapshot_download

print("⏳ Downloading default OmniVoice TTS model...")
tts_path = snapshot_download("k2-fsa/OmniVoice")
print(f"OmniVoice cached at: {tts_path}")

print(
    "\n⏳ Downloading Faster-Whisper (base) for automatic reference transcription..."
)
asr_path = snapshot_download("Systran/faster-whisper-base")
print(f"Faster-Whisper cached at: {asr_path}")

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
import urllib.request

PORT = 3900
REPO_DIR = "/kaggle/working/VoiceStudio"
LOG_PATH = "/kaggle/working/backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

# 1. Clean up any existing processes on port 3900
subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(1)

# 2. Set server environment variables
env = os.environ.copy()
env["OMNIVOICE_SERVER_MODE"] = "1"
env["OMNIVOICE_ALLOWED_HOSTS"] = "*"
env["OMNIVOICE_DATA_DIR"] = "/kaggle/working/omnivoice_data"
env["PYTHONUNBUFFERED"] = "1"

# 3. Launch Uvicorn Backend
log_file = open(LOG_PATH, "wb")
backend_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--app-dir",
        "backend",
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
    ],
    cwd=REPO_DIR,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(
    f"Starting VoiceStudio Backend (PID {backend_proc.pid}). Waiting for health check..."
)

# 4. Wait until the backend reports healthy
started = False
for _ in range(60):
    if backend_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=3) as response:
            info = json.load(response)
            print(f"\n Backend is LIVE: {info}")
            started = True
            break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(3)

if not started:
    with open(LOG_PATH, "r", errors="replace") as f:
        print("\nBackend failed to start. Logs:\n", "".join(f.readlines()[-40:]))
    raise SystemExit("Backend startup failed.")

# 5. Start Cloudflare Tunnel and display the public URL
print("\nStarting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\n" + "=" * 65)
        print(f"VOICESTUDIO WEB UI IS READY!")
        print(f"Click here to open: {public_url}")
        print("=" * 65 + "\n")
        break

# Keep the cell alive in the tutorial so the server doesn't shut down
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("Shutting down server and tunnel...")
    tunnel_proc.terminate()
    backend_proc.terminate()